In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import asyncio
import pathlib
import time

import numpy as np
from PIL import Image

In [3]:
im = Image.open(pathlib.Path("bindings") / "src" / "waterfall.jpg")
specgram_linear = 10**((67.7 * np.asarray(im, dtype=np.float32) / 255 + 20) / 10)

In [4]:
from maia_waterfall_widget import Waterfall, WaterfallShape
w_shape = WaterfallShape(time=512, freq=specgram_linear.shape[1])
w = Waterfall(w_shape)
w

In [5]:
import ipywidgets as widgets
spectrum_visible = widgets.Checkbox(
    value=False,
    description="Show spectrum",
)
widgets.jslink((spectrum_visible, 'value'), (w, 'spectrum_visible'))
waterfall_visible = widgets.Checkbox(
    value=True,
    description="Show waterfall",
)
widgets.jslink((waterfall_visible, 'value'), (w, 'waterfall_visible'))
cmap_select = widgets.Dropdown(
    options=['viridis', 'turbo', 'inferno'],
    value='turbo',
    description="Colormap:",
)
widgets.link((cmap_select, 'value'), (w, 'colormap'))
min_max_db = widgets.IntRangeSlider(
    value=[25, 95],
    min=0,
    max=120,
    step=1,
    continuous_update=False,
    description="Scale (dB):",
)
widgets.link((min_max_db, 'value'), (w, 'waterfall_min_db'), transform=[
    lambda src: src[0],
    lambda tgt: (tgt, min_max_db.value[1]),
])
widgets.link((min_max_db, 'value'), (w, 'waterfall_max_db'), transform=[
    lambda src: src[1],
    lambda tgt: (min_max_db.value[0], tgt),
])
widgets.HBox([waterfall_visible, spectrum_visible, cmap_select, min_max_db])

In [6]:
async def put_spectra():
    while True:
        for spectrum_linear in specgram_linear:
            w.put_spectrum(spectrum_linear)
            await asyncio.sleep(.034)
## can optionally run the task without blocking, but then you need to explicitly .cancel() it
# put_spectra_task = asyncio.create_task(put_spectra())
# put_spectra_task

In [ ]:
try:
    async with asyncio.TaskGroup() as group:
        group.create_task(put_spectra())
except asyncio.CancelledError:
    pass